In [ ]:
import pandas as pd
import pm4py

# ==========================================
# PHASE 1: DATA PREPARATION (CRISP-DM)
# ==========================================
print("Loading event log...")
# Load the CSV file generated by the Mesa simulation
df = pd.read_csv("bitcoin_agent_event_log.csv")

# Format timestamps properly
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# PM4Py requires specific column names to understand what a Case, Activity, and Timestamp are.
df = pm4py.format_dataframe(
    df, 
    case_id='CaseID', 
    activity_key='Activity', 
    timestamp_key='Timestamp'
)

# Convert the pandas dataframe into a PM4Py Event Log object
log = pm4py.convert_to_event_log(df)
print(f"Log loaded successfully! Total events: {len(df)}")

# ==========================================
# PHASE 2: MODELING (PROCESS DISCOVERY)
# ==========================================
print("\nGenerating the complete Directly-Follows Graph (Process Map)...")
dfg, start_activities, end_activities = pm4py.discover_dfg(log)

# SAVING THE FILE INSTEAD OF VIEWING IT
pm4py.save_vis_dfg(dfg, start_activities, end_activities, "complete_process_map.png")
print("--> Saved complete process map as 'complete_process_map.png'")


# ==========================================
# PHASE 3: VARIANT ANALYSIS & OUTLIER DETECTION
# ==========================================
print("\n--- Variant Analysis (Finding the Outliers) ---")
# A "variant" is a unique path from start to finish. 
variants = pm4py.get_variants_as_tuples(log)
variant_counts = {variant: len(cases) for variant, cases in variants.items()}

# Sort variants by frequency (highest to lowest)
sorted_variants = sorted(variant_counts.items(), key=lambda item: item[1], reverse=True)

print(f"Total unique behavioral variants discovered: {len(sorted_variants)}")
print("\nTop 3 MOST COMMON Variants (The 'Ideal' Process Paths):")
for variant, count in sorted_variants[:3]:
    print(f"- Frequency: {count} cases | Path: {variant[:3]}... (truncated)")

print("\nTop 3 RAREST Variants (The Outliers/Deviations):")
for variant, count in sorted_variants[-3:]:
    print(f"- Frequency: {count} cases | Path: {variant[:3]}... (truncated)")


# ==========================================
# PHASE 4: FILTERING (BEMTHUIS METHODOLOGY)
# ==========================================
# Filtering isolates specific behaviors for face validity checks, 
# preventing "spaghetti" models.

print("\n--- Applying Filters ---")

# Filter 1: Isolate specific anomalous activities (e.g., Orphaned Blocks and Crashes)
suspicious_activities = ["Block_Orphaned", "Hardware_Crash_honest", "Hardware_Crash_selfish"]
filtered_outlier_log = pm4py.filter_event_attribute_values(
    log, 
    "concept:name", 
    suspicious_activities, 
    level="case", 
    retain=True
)
print(f"Cases containing Outlier Events (Crashes/Orphans): {len(filtered_outlier_log)}")

if len(filtered_outlier_log) > 0:
    print("Generating Process Map for Outlier Cases Only...")
    dfg_filtered, start_filtered, end_filtered = pm4py.discover_dfg(filtered_outlier_log)
    
    # SAVING THE FILTERED FILE
    pm4py.save_vis_dfg(dfg_filtered, start_filtered, end_filtered, "filtered_outlier_map.png")
    print("--> Saved filtered outlier map as 'filtered_outlier_map.png'")

# Filter 2: Heuristics Net
# The Heuristics miner is excellent for handling noisy, complex agent logs
print("\nGenerating Heuristics Net...")
heu_net = pm4py.discover_heuristics_net(log)

# SAVING THE HEURISTICS NET
pm4py.save_vis_heuristics_net(heu_net, "heuristics_net.png")
print("--> Saved heuristics net as 'heuristics_net.png'")

print("\nProcess Mining Analysis Complete! Check your folder for the PNG files.")